In [1]:
import pandas as pd
import numpy as np
import nfl_data_py as nfl
from pygam import LinearGAM, s, f, te
import matplotlib.pyplot as plt
import data_processing_functions as helpy
import manual_ml_stuff as ml
import lightgbm as lgb
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

In [2]:
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

In [3]:
recieving_2024 = helpy.get_year_table(2024)
recieving_2023 = helpy.get_year_table(2023)
recieving_2022 = helpy.get_year_table(2022)
recieving_2021 = helpy.get_year_table(2021)
recieving_2020 = helpy.get_year_table(2020)
recieving_2019 = helpy.get_year_table(2019)
recieving_2018 = helpy.get_year_table(2018)

In [4]:
recieving = helpy.merge_datasets([recieving_2018, recieving_2019, recieving_2020, 
                                  recieving_2021, recieving_2022, recieving_2023, recieving_2024])

skills in order: efficency vs man and zone seperately, threat to all parts of the field, drops, positional value, translatability 

In [5]:
X = recieving[['man_yprr', 'man_avg_depth_of_target', 'man_yards_per_game', 'zone_yprr', 'zone_avg_depth_of_target', 'zone_yards_per_game', 'is_power_four', 
               'var_depth', 'deep_yards', 'perc_deep_yards', 'drop_rate', 'wide_rate']]
    
y = recieving['pick']

In [6]:
X['is_power_four'] = X['is_power_four'].astype(np.float32)

/tmp/ipykernel_32713/2589465502.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X['is_power_four'] = X['is_power_four'].astype(np.float32)


In [7]:
X_norm = (X - X.min()) / (X.max() - X.min())

In [9]:
# Convert to PyTorch tensors
X_tensor = torch.tensor(X_norm.values, dtype=torch.float32)
y_tensor = torch.tensor(y.values, dtype=torch.float32).unsqueeze(1)

In [10]:
# Dataset & DataLoader
dataset = TensorDataset(X_tensor, y_tensor)
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

In [11]:
model = ml.NonMonotoneMultiplicativeModel()
loss_fn = nn.MSELoss()  # regression
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

In [12]:
n_epochs = 100

for epoch in range(n_epochs):
    epoch_loss = 0
    for x_batch, y_batch in dataloader:
        print(model(x_batch))
        optimizer.zero_grad()              # Step 1: reset gradients
        y_pred = model(x_batch)            # Step 2: forward pass
        loss = loss_fn(y_pred, y_batch)   # Step 3: compute loss
        loss.backward()                    # Step 4: backward pass (compute gradients)
        optimizer.step()                   # Step 5: update weights
        epoch_loss += loss.item() * x_batch.size(0)
    
    epoch_loss /= len(dataloader.dataset)
    print(f"Epoch {epoch+1}/{n_epochs}, Loss: {epoch_loss:.4f}")

tensor([[0.4215],
        [0.4509],
        [0.4216],
        [0.4391],
        [0.4376],
        [0.4611],
        [0.4320],
        [0.4233],
        [0.4571],
        [0.4283],
        [0.4535],
        [0.4436],
        [0.4663],
        [0.4637],
        [0.4443],
        [0.4621],
        [0.4656],
        [0.4538],
        [0.4340],
        [0.4429],
        [0.4389],
        [0.4273],
        [0.4404],
        [0.4365],
        [0.4480],
        [0.4520],
        [0.4374],
        [0.4368],
        [0.3980],
        [0.4492],
        [0.4187],
        [0.4467]], grad_fn=<SubBackward0>)
tensor([[0.4598],
        [0.4824],
        [0.4638],
        [0.4139],
        [0.4790],
        [0.4535],
        [0.4587],
        [0.4558],
        [0.4562],
        [0.4341],
        [0.4644],
        [0.4561],
        [0.4649],
        [0.4746],
        [0.4373],
        [0.4640],
        [0.4657],
        [0.4658],
        [0.4752],
        [0.4597],
        [0.4620],
        [0.4493],
   

In [18]:
model.eval()
with torch.no_grad():  # no gradients needed for prediction
    y_pred = model(X_tensor)  # X: tensor of shape (num_samples, 11)

In [19]:
y_pred

tensor([[ 51.7202],
        [227.8466],
        [ 74.0462],
        [151.2871],
        [ 92.9830],
        [103.8216],
        [ 69.6747],
        [ 98.0588],
        [146.2706],
        [219.7998],
        [166.9394],
        [ 46.5206],
        [ 28.3946],
        [117.7051],
        [109.7598],
        [159.8332],
        [149.6034],
        [192.3097],
        [122.6815],
        [200.1170],
        [165.7390],
        [157.5831],
        [137.1935],
        [184.9346],
        [ 78.9062],
        [110.8406],
        [ 69.0779],
        [ 67.6407],
        [244.4056],
        [117.2037],
        [172.0784],
        [ 56.2855],
        [100.8164],
        [116.2242],
        [171.1381],
        [ 63.7378],
        [118.4533],
        [ 97.5292],
        [109.4207],
        [230.2691],
        [185.8311],
        [119.4478],
        [ 36.2751],
        [205.2346],
        [ 44.0062],
        [223.0745],
        [153.6885],
        [164.6186],
        [164.4760],
        [179.1934],


In [15]:
recieving['predicted_slot'] = y_pred

In [16]:
recieving

,player,player_id,position,team_name,player_game_count,contested_receptions,contested_catch_rate,targets,yards,touchdowns,avg_depth_of_target,drop_rate,wide_rate,behind_los_yards,short_yards,medium_yards,deep_yards,man_targets,man_yprr,man_avg_depth_of_target,man_yards,zone_targets,zone_yprr,zone_avg_depth_of_target,zone_yards,pick,pfr_player_name,college,var_depth,perc_deep_yards,yards_per_game,man_yards_per_game,zone_yards_per_game,is_power_four,predicted_slot
0,Andy Isabella,47448,WR,UMASS,12,9,36.0,146,1696,13,12.1,5.6,55.8,170,351,470,705,39,3.89,14.2,428,100,4.47,11.5,1216,62,Andy Isabella,Massachusetts,50307.333333,0.415684,141.333333,35.666667,101.333333,False,51.720219
1,John Ursua,26586,WR,HAWAII,13,10,47.6,145,1343,16,12.1,12.7,3.0,21,513,386,423,38,2.81,11.4,427,94,2.27,12.9,886,236,John Ursua,Hawaii,46874.250000,0.314966,103.307692,32.846154,68.153846,False,227.846558
2,Dillon Mitchell,42338,WR,OREGON,13,10,27.8,130,1184,10,14.6,9.6,76.5,90,321,264,509,59,3.65,14.6,533,66,2.58,14.0,618,239,Dillon Mitchell,Oregon,29818.000000,0.429899,91.076923,41.000000,47.538462,True,74.046227
3,KeeSean Johnson,39587,WR,FRESNO ST,14,11,47.8,129,1345,8,11.1,6.8,69.6,101,389,314,541,29,2.82,8.4,282,98,3.28,11.9,1052,174,KeeSean Johnson,Fresno St.,33514.250000,0.402230,96.071429,20.142857,75.142857,False,151.287109
4,Kelvin Harmon,47931,WR,NC STATE,12,17,56.7,117,1186,7,15.1,4.7,96.4,31,302,387,466,30,3.27,12.4,363,81,2.85,15.9,804,206,Kelvin Harmon,North Carolina St.,35813.666667,0.392917,98.833333,30.250000,67.000000,True,92.983009
5,A.J. Brown,48327,WR,OLE MISS,12,8,40.0,115,1307,6,11.2,5.6,40.4,104,457,359,387,39,2.75,13.8,432,70,3.28,9.7,819,51,A.J. Brown,Mississippi,23750.916667,0.296098,108.916667,36.000000,68.250000,True,103.821602
6,N'Keal Harry,48297,WR,ARIZONA ST,12,19,51.4,113,1088,9,10.4,6.4,66.5,213,213,386,276,49,4.18,10.7,568,58,1.79,10.4,445,32,N'Keal Harry,Arizona St.,6658.000000,0.253676,90.666667,47.333333,37.083333,True,69.674660
7,Parris Campbell,47940,WR,OHIO STATE,14,5,71.4,111,1071,12,4.7,5.2,11.5,441,408,169,53,20,3.48,5.8,223,84,3.55,4.5,805,59,Parris Campbell,Ohio St.,35184.916667,0.049486,76.500000,15.928571,57.500000,True,98.058762
8,Scott Miller,47481,WR,BOWL GREEN,11,6,37.5,107,1137,9,12.3,6.7,16.3,0,368,390,379,36,2.98,14.0,298,66,3.24,11.6,796,208,Scott Miller,Bowling Green,35990.916667,0.333333,103.363636,27.090909,72.363636,False,146.270599
9,Travis Fulgham,48023,WR,DOMINION,12,15,57.7,107,1083,9,15.6,7.4,88.8,0,152,359,572,27,2.57,15.4,247,71,2.34,16.5,827,184,Travis Fulgham,Old Dominion,61982.250000,0.528163,90.250000,20.583333,68.916667,False,219.799835


In [17]:
print(y_tensor)

tensor([[ 62.],
        [236.],
        [239.],
        [174.],
        [206.],
        [ 51.],
        [ 32.],
        [ 59.],
        [208.],
        [184.],
        [ 67.],
        [103.],
        [ 25.],
        [ 93.],
        [ 36.],
        [ 66.],
        [203.],
        [247.],
        [171.],
        [149.],
        [126.],
        [ 56.],
        [ 76.],
        [187.],
        [ 64.],
        [ 22.],
        [ 92.],
        [161.],
        [207.],
        [ 59.],
        [151.],
        [ 15.],
        [ 49.],
        [ 81.],
        [200.],
        [ 25.],
        [217.],
        [ 46.],
        [166.],
        [212.],
        [168.],
        [ 21.],
        [ 17.],
        [173.],
        [ 33.],
        [176.],
        [220.],
        [ 57.],
        [187.],
        [165.],
        [214.],
        [252.],
        [ 10.],
        [129.],
        [ 34.],
        [219.],
        [ 85.],
        [131.],
        [ 91.],
        [ 20.],
        [204.],
        [ 82.],
        